# 香港夏季天气数据整理与指标计算

本 Notebook 对应项目总体计划的前两步：从香港天文台原始 CSV 生成逐日清洗表、完整性表、年度指标、月度指标，以及早期与近期汇总表。

In [1]:
from pathlib import Path

import polars as pl

from hk_summer_weather.pipeline import PipelinePaths, run_pipeline

current_directory = Path.cwd()
project_root = (
    current_directory
    if (current_directory / "pyproject.toml").exists()
    else current_directory.parent
)
paths = PipelinePaths(
    raw=project_root / "data/raw",
    processed=project_root / "data/processed",
    metrics=project_root / "data/metrics",
)
result = run_pipeline(paths)
result

PipelineResult(daily_rows=3220, valid_years=35, annual_rows=35, monthly_rows=105)

## 数据完整性

完整夏季要求 6–8 月的 92 天中，日均温、日最高温和日雨量全部存在，且香港天文台完整性标记均为 `C`。

In [2]:
quality = pl.read_csv(paths.processed / "summer_data_quality_by_year.csv")
quality

year,expected_days,mean_temp_complete_days,max_temp_complete_days,rainfall_complete_days,complete_days,trace_days,valid_summer
i64,i64,i64,i64,i64,i64,i64,bool
1991,92,92,92,92,92,16,true
1992,92,92,92,92,92,19,true
1993,92,92,92,92,92,18,true
1994,92,92,92,92,92,11,true
1995,92,92,92,92,92,12,true
…,…,…,…,…,…,…,…
2021,92,92,92,92,92,14,true
2022,92,92,92,92,92,7,true
2023,92,92,92,92,92,11,true


## 年度与月度指标

In [3]:
annual = pl.read_csv(paths.metrics / "annual_summer_metrics.csv")
monthly = pl.read_csv(paths.metrics / "monthly_summer_metrics.csv")
annual.head(), monthly.head()

(shape: (5, 6)
 ┌──────┬────────────────────┬───────────────────┬─────────────────┬───────────────────┬────────────┐
 │ year ┆ summer_mean_temp_c ┆ summer_total_rain ┆ hot_days_ge_33c ┆ rainy_days_ge_0_1 ┆ valid_days │
 │ ---  ┆ ---                ┆ fall_mm           ┆ ---             ┆ mm                ┆ ---        │
 │ i64  ┆ f64                ┆ ---               ┆ i64             ┆ ---               ┆ i64        │
 │      ┆                    ┆ f64               ┆                 ┆ i64               ┆            │
 ╞══════╪════════════════════╪═══════════════════╪═════════════════╪═══════════════════╪════════════╡
 │ 1991 ┆ 28.596739          ┆ 967.6             ┆ 9               ┆ 55                ┆ 92         │
 │ 1992 ┆ 28.257609          ┆ 988.6             ┆ 8               ┆ 47                ┆ 92         │
 │ 1993 ┆ 28.792391          ┆ 881.7             ┆ 6               ┆ 55                ┆ 92         │
 │ 1994 ┆ 27.91087           ┆ 2035.0            ┆ 5               

## 早期与近期汇总

月雨量是各年份相同月份总雨量的平均值，不是十年累计量。

In [4]:
period_annual = pl.read_csv(paths.metrics / "period_annual_summary.csv")
period_monthly = pl.read_csv(paths.metrics / "period_monthly_summary.csv")
period_annual, period_monthly

(shape: (2, 6)
 ┌─────────────────┬────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
 │ period          ┆ year_count ┆ average_summer ┆ average_summer ┆ average_hot_da ┆ average_rainy_ │
 │ ---             ┆ ---        ┆ _mean_temp_c   ┆ _total_rainfal ┆ ys_ge_33c      ┆ days_ge_0_1mm  │
 │ str             ┆ i64        ┆ ---            ┆ l_…            ┆ ---            ┆ ---            │
 │                 ┆            ┆ f64            ┆ ---            ┆ f64            ┆ f64            │
 │                 ┆            ┆                ┆ f64            ┆                ┆                │
 ╞═════════════════╪════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
 │ early_1991_2000 ┆ 10         ┆ 28.466196      ┆ 1414.51        ┆ 7.0            ┆ 56.6           │
 │ recent_2016_202 ┆ 10         ┆ 29.228696      ┆ 1257.79        ┆ 31.2           ┆ 60.5           │
 │ 5               ┆            ┆                ┆                ┆